# Desafio de Regressão: Previsão de Valores de Imóveis com Perceptron Multicamadas (MLP)

**Tarefa:** Utilizar um modelo **Perceptron Multicamadas (MLP)** para realizar uma **tarefa de regressão** no **Dataset California Housing**.

**Objetivo:** Treinar o MLP para prever o valor mediano das casas (`median_house_value`) em distritos da Califórnia, com base em suas características.

---

## O Dataset California Housing

Dataset clássico de aprendizado de máquina, ideal para introdução a regressão e redes neurais.

**Disponibilidade:** O dataset pode ser acessado diretamente no **Google Colab**, geralmente dentro do diretório `sample_data`.

## Atributos do Dataset

A tabela abaixo descreve as *features* (entradas) e a variável alvo (saída) para o modelo MLP:

| Atributo | Descrição | Variável |
| :--- | :--- | :--- |
| `longitude` | Medida de quão oeste o distrito está. | Entrada |
| `latitude` | Medida de quão ao norte o distrito está. | Entrada |
| `housing_median_age` | Idade mediana das casas no distrito. | Entrada |
| `total_rooms` | Número total de cômodos. | Entrada |
| `total_bedrooms` | Número total de quartos. | Entrada |
| `population` | População total. | Entrada |
| `households` | Número total de famílias. | Entrada |
| `median_income` | Renda mediana das famílias (em dezenas de milhar de dólares). | Entrada |
| **`median_house_value`** | **Valor mediano das casas (em dólares).** | **Alvo (Saída)** |

## Diretrizes para o MLP

1.  **Pré-processamento:** É obrigatório aplicar **escalonamento (Padronização ou Normalização)** aos dados de entrada, pois o MLP é sensível à magnitude das *features*.
2.  **Arquitetura:** Construir uma Rede Neural MLP com pelo menos uma camada oculta.
3.  **Configuração de Ativação:** O uso da função **ReLU** nas camadas ocultas é sugerido por ser um bom ponto de partida. **No entanto, o aluno é livre para experimentar outras funções de ativação, como Sigmoide ou Tanh, para comparar resultados.**
4.  **Função de Perda:** Utilizar a função de perda **MSE** (Erro Quadrático Médio) para otimização.
5.  **Avaliação:** Avaliar a performance do modelo usando o conjunto de teste, focando na métrica **RMSE** (Raiz do Erro Quadrático Médio).

In [1]:
!pip install numpy torch torchsummary scikit-learn pandas plotly

In [2]:
# Importar bibliotecas necessárias

# Deep Learning / Machine Learning

import numpy as np
import torch
import torch.nn as nn
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# EDA
import pandas as pd
import plotly.express as px

In [3]:
# Carregar o dataset
df_housing = pd.read_csv('/content/sample_data/california_housing_train.csv')
df_housing_test = pd.read_csv('/content/sample_data/california_housing_train.csv')

In [4]:
# Visualizar primeiras linhas
df_housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


In [5]:
# Estatísticas
df_housing.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000,17000.000000
mean,-119.562108,35.625225,28.589353,2643.664412,539.410824,1429.573941,501.221941,3.883578,207300.912353
std,2.005166,2.137340,12.586937,2179.947071,421.499452,1147.852959,384.520841,1.908157,115983.764387
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.790000,33.930000,18.000000,1462.000000,297.000000,790.000000,282.000000,2.566375,119400.000000
50%,-118.490000,34.250000,29.000000,2127.000000,434.000000,1167.000000,409.000000,3.544600,180400.000000
75%,-118.000000,37.720000,37.000000,3151.250000,648.250000,1721.000000,605.250000,4.767000,265000.000000
max,-114.310000,41.950000,52.000000,37937.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


# Preparação de Dados para EDA

In [6]:
numerical_features = df_housing.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [7]:
numerical_features.remove('median_house_value')

In [8]:
# Variável target
target = ['median_house_value']

# EDA

In [9]:
# Distribuição da variável, target, usando plotly
fig = px.histogram(df_housing, x=target, nbins=50, title='Distribuição median_house_value')
fig.show()

In [10]:
# Distribuição das variáveis numéricas
for feature in numerical_features:
    fig = px.histogram(df_housing, x=feature, nbins=50, title=f'Distribuição de {feature}')
    fig.show()

In [11]:
# BoxPlot das variáveis numéricas
for feature in numerical_features:
    fig = px.box(df_housing, y=feature, title=f'BoxPlot de {feature}')
    fig.show()

## Preparar dados para Correlações

In [12]:
# Mostrar correlação entre as variáveis, usando plotly
fig = px.imshow(df_housing.corr(), text_auto=True, aspect="auto",title='Correlação entre as variáveis', width=1080, height=900)
fig.show()

# Preparar dados para treinamento da rede neural

In [13]:
# Dividir o dataset entre X e y
X = df_housing.drop(columns=target, axis=1)
y = np.array(df_housing[target])

In [14]:
X

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250
...,...,...,...,...,...,...,...,...
16995,-124.26,40.58,52.0,2217.0,394.0,907.0,369.0,2.3571
16996,-124.27,40.69,36.0,2349.0,528.0,1194.0,465.0,2.5179
16997,-124.30,41.84,17.0,2677.0,531.0,1244.0,456.0,3.0313
16998,-124.30,41.80,19.0,2672.0,552.0,1298.0,478.0,1.9797


In [15]:
y

array([[ 66900.],
       [ 80100.],
       [ 85700.],
       ...,
       [103600.],
       [ 85800.],
       [ 94600.]])

In [16]:
# Dividir entre treino e validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.5, random_state=42, shuffle=True)

In [17]:
# Grupos de teste
X_test = df_housing_test.drop(columns=target, axis=1)
y_test = np.array(df_housing_test[target])

In [18]:
# Aplicar transformação por tipo
numeric_transformer = MinMaxScaler()

In [19]:
# Aplicar transformação por coluna
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
    ]
)

In [20]:
# Aplicar preprocessor nos splits
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

scaler_y = MinMaxScaler()
y_train = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_val = scaler_y.transform(y_val.reshape(-1, 1))
y_test = scaler_y.transform(y_test.reshape(-1, 1))

In [21]:
# Shape dos datasets
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}')
print(f'y_val: {y_val.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (8500, 8)
y_train: (8500, 1)
X_val: (8500, 8)
y_val: (8500, 1)
X_test: (17000, 8)
y_test: (17000, 1)


In [22]:
# Criar uma estrutura do dataset de Housing em uma classe Dataset do Pytorch
class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [23]:
# Criar Datasets
dataset_train = HousingDataset(X_train, y_train)
dataset_val = HousingDataset(X_val, y_val)
dataset_test = HousingDataset(X_test, y_test)

In [24]:
# Criar Dataloaders
dataloader_train = DataLoader(dataset_train, batch_size=32, drop_last=True)
dataloader_val = DataLoader(dataset_val, batch_size=32, drop_last=True)
dataloader_test = DataLoader(dataset_test, batch_size=32, drop_last=True)

# Definir arquitetura da rede

In [25]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_layer_sizes=[128, 64, 32, 16], output_size=1, dropout_rate=0.2):
      super(NeuralNetwork, self).__init__()
      self.layer1 = nn.Linear(input_size, hidden_layer_sizes[0])
      self.bn1 = nn.BatchNorm1d(hidden_layer_sizes[0])
      self.dropout1 = nn.Dropout(dropout_rate)

      self.layer2 = nn.Linear(hidden_layer_sizes[0], hidden_layer_sizes[1])
      self.bn2 = nn.BatchNorm1d(hidden_layer_sizes[1])
      self.dropout2 = nn.Dropout(dropout_rate)

      self.layer3 = nn.Linear(hidden_layer_sizes[1], hidden_layer_sizes[2])
      self.bn3 = nn.BatchNorm1d(hidden_layer_sizes[2])
      self.dropout3 = nn.Dropout(dropout_rate)

      self.layer4 = nn.Linear(hidden_layer_sizes[2], hidden_layer_sizes[3])
      self.bn4 = nn.BatchNorm1d(hidden_layer_sizes[3])
      self.dropout4 = nn.Dropout(dropout_rate)

      self.output_layer = nn.Linear(hidden_layer_sizes[3], output_size)
      self.relu = nn.ReLU()

    def forward(self, x):
      x = self.relu(self.bn1(self.layer1(x)))
      x = self.dropout1(x)

      x = self.relu(self.bn2(self.layer2(x)))
      x = self.dropout2(x)

      x = self.relu(self.bn3(self.layer3(x)))
      x = self.dropout3(x)

      x = self.relu(self.bn4(self.layer4(x)))
      x = self.dropout4(x)

      x = self.output_layer(x)
      return x

In [26]:
# Instanciar o modelo
model = NeuralNetwork(input_size=X_train.shape[1], hidden_layer_sizes=[64, 32, 16, 8], output_size=1)

In [27]:
# Visualizar a arquitetura do modelo
summary(model, (X_train.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]             576
       BatchNorm1d-2                   [-1, 64]             128
              ReLU-3                   [-1, 64]               0
           Dropout-4                   [-1, 64]               0
            Linear-5                   [-1, 32]           2,080
       BatchNorm1d-6                   [-1, 32]              64
              ReLU-7                   [-1, 32]               0
           Dropout-8                   [-1, 32]               0
            Linear-9                   [-1, 16]             528
      BatchNorm1d-10                   [-1, 16]              32
             ReLU-11                   [-1, 16]               0
          Dropout-12                   [-1, 16]               0
           Linear-13                    [-1, 8]             136
      BatchNorm1d-14                   

# Treinar a Rede Neural

In [28]:
# treinar a rede
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

NUM_EPOCHS = 1000
train_losses = []
val_losses = []

# parametros do early stopping
patience = 100
min_delta = 1e-6
best_val_loss = float('inf')
epochs_no_improve = 0
best_epoch = -1

for epoch in range(NUM_EPOCHS):
  model.train()
  running_train_loss = 0.0
  for data in dataloader_train:
    # Zerar os gradientes
    optimizer.zero_grad()

    # Dividir entre input e output
    inputs, targets = data

    # Forward pass
    outputs = model(inputs)

    # Calcular a perda
    loss = criterion(outputs, targets)

    # Backward pass
    loss.backward()

    # Gradient Cliping
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # Atualizar os pesos
    optimizer.step()

    running_train_loss += loss.item()

  epoch_train_loss = running_train_loss / len(dataloader_train)
  train_losses.append(epoch_train_loss)

  # Fase de validação
  model.eval()
  running_val_loss = 0.0
  with torch.no_grad():
    for data in dataloader_val:
      # Dividir entre input e output
      inputs, targets = data

      # Forward Pass
      outputs = model(inputs)

      # Calcular a perda
      loss = criterion(outputs, targets)
      running_val_loss += loss.item()

  epoch_val_loss = running_val_loss / len(dataloader_val)
  val_losses.append(epoch_val_loss)

  # Checar improvement para Early Stopping
  if best_val_loss - epoch_val_loss > min_delta:
    best_val_loss = epoch_val_loss
    best_epoch = epoch
    epochs_no_improve = 0
  else:
    epochs_no_improve += 1

  if epoch % 10 == 0:
    print(f'Epoch {epoch}, Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}')

  if epochs_no_improve >= patience:
    print(f'Early stopping! melhor epoch: {best_epoch} - Vall Loss: {best_val_loss:.6f}')
    break

Epoch 0, Train Loss: 0.061247, Val Loss: 0.021144
Epoch 10, Train Loss: 0.022809, Val Loss: 0.017282
Epoch 20, Train Loss: 0.022158, Val Loss: 0.016431
Epoch 30, Train Loss: 0.022097, Val Loss: 0.016028
Epoch 40, Train Loss: 0.021532, Val Loss: 0.015971
Epoch 50, Train Loss: 0.021996, Val Loss: 0.015528
Epoch 60, Train Loss: 0.021109, Val Loss: 0.015152
Epoch 70, Train Loss: 0.020550, Val Loss: 0.015415
Epoch 80, Train Loss: 0.020098, Val Loss: 0.015903
Epoch 90, Train Loss: 0.020456, Val Loss: 0.016188
Epoch 100, Train Loss: 0.020457, Val Loss: 0.014991
Epoch 110, Train Loss: 0.020545, Val Loss: 0.015332
Epoch 120, Train Loss: 0.020349, Val Loss: 0.015449
Epoch 130, Train Loss: 0.020102, Val Loss: 0.015957
Epoch 140, Train Loss: 0.019889, Val Loss: 0.015083
Epoch 150, Train Loss: 0.019365, Val Loss: 0.015182
Epoch 160, Train Loss: 0.020117, Val Loss: 0.015853
Epoch 170, Train Loss: 0.019594, Val Loss: 0.015023
Epoch 180, Train Loss: 0.019772, Val Loss: 0.016339
Epoch 190, Train Loss: 

# Visualizar resultados do treinamento

In [29]:
# Plotar loss de treino e validação com Plotly
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, NUM_EPOCHS+1)), y=train_losses, mode='lines', name='Train Loss'))
fig.add_trace(go.Scatter(x=list(range(1, NUM_EPOCHS+1)), y=val_losses, mode='lines', name='Val Loss'))
fig.add_vline(x=best_epoch + 1, line_dash="dash", line_color="red", annotation_text=f"Best Epoch: {best_epoch+1}", annotation_position="top right")
fig.update_layout(title='Loss de Treino e Validação', xaxis_title='Epoch', yaxis_title='Loss')
fig.show()

# Validar Loss no conjunto de testes

In [30]:
# Fase de teste
model.eval()
running_test_loss = 0.0
with torch.no_grad():
  for data in dataloader_test:
    # Dividir entre input e output
    inputs, targets = data

    # Forward Pass
    outputs = model(inputs)

    # Calcular a perda
    loss = criterion(outputs, targets)
    running_test_loss += loss.item()

epoch_test_loss = running_test_loss / len(dataloader_test)

In [31]:
# Comparar Loss Treino, Val e teste
train_losses[best_epoch], val_losses[best_epoch], epoch_test_loss

(0.020050887568449638, 0.0144852114279034, 0.015787746631645592)